# Phase 26 — Component Composition Diagnosis (Phase 18)
## NeuroForge Experimental Research

Phase 17: branch info survives fusion; RC-agree ~100% vs RC-disagree ~0-2%. Question: WHY does the predictor select one available component when the task requires combining components? Decision-semantic diagnosis across: label semantics, component vs joint prediction, representation availability, head behavior, loss incentives, calibration, RC-vs-FRC structure. RC uses R+C semantics per construction (F is a negative control).

## 1. Phase 17 baseline

In [1]:
import json
p18 = json.load(open('../results/metrics/phase18_component_composition/summary.json', encoding='utf-8'))
for k in ('baseline_perf_mean','depth3_perf_mean'):
    d = p18[k]
    print(f"{k}: " + '  '.join(f"{f}={d.get(f,0)*100:.1f}%" for f in ('F','R','C','FR','RC','FC','FRC')))
print('Phase 17: info survives fusion; RC-disagree collapses.')

baseline_perf_mean: F=100.0%  R=57.8%  C=98.9%  FR=76.9%  RC=53.9%  FC=49.4%  FRC=79.4%
depth3_perf_mean: F=100.0%  R=62.5%  C=99.2%  FR=76.7%  RC=53.6%  FC=50.0%  FRC=80.0%
Phase 17: info survives fusion; RC-disagree collapses.


## 2. Research question

In [2]:
print('Why select one available component when the task requires combining?')
print('Distinguish: semantics / component-vs-joint / availability / head / loss / calibration / RC-vs-FRC.')

Why select one available component when the task requires combining?
Distinguish: semantics / component-vs-joint / availability / head / loss / calibration / RC-vs-FRC.


## 3. Component target audit (18B: diagnostic metadata, official labels untouched)

In [3]:
import csv
n = 0
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/component_targets.csv')):
    n += 1
    if n <= 3:
        print(row)
print(f'... total RC/FRC metadata rows: {n}')
print('RC components are R+C per construction (F is a negative control here).')

{'seed': '11', 'sample': '3', 'family': 'RC', 'sf': '1', 'sr': '1', 'sc': '1', 'target': '1'}
{'seed': '11', 'sample': '9', 'family': 'FRC', 'sf': '1', 'sr': '-1', 'sc': '1', 'target': '1'}
{'seed': '11', 'sample': '10', 'family': 'RC', 'sf': '-1', 'sr': '1', 'sc': '1', 'target': '1'}
... total RC/FRC metadata rows: 720
RC components are R+C per construction (F is a negative control here).


## 4. Component prediction matrix (18C: rep x component, incl. F negative control)

In [4]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/component_probe_matrix.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"{row['representation']:>6}: " + '  '.join(f"{k}={float(v)*100:.1f}%" for k,v in row.items() if k not in ('seed','representation')))
print('F-on-RC must read ~chance (no F injected): validates probes are honest.')

  feat: F=64.3%  F_on_RC=45.8%  R=50.0%  R_on_RC=40.8%  C=63.6%  C_on_RC=51.7%
   rel: F=65.7%  F_on_RC=50.0%  R=48.9%  R_on_RC=40.0%  C=63.9%  C_on_RC=55.0%
   ctx: F=43.9%  F_on_RC=47.5%  R=48.8%  R_on_RC=40.8%  C=65.8%  C_on_RC=100.0%
 fused: F=64.5%  F_on_RC=47.5%  R=49.0%  R_on_RC=42.5%  C=71.9%  C_on_RC=66.7%
F-on-RC must read ~chance (no F injected): validates probes are honest.


## 5. Joint decodability (18D: 4-class (sr,sc) target on RC)

In [5]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/joint_decodability.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"{row['representation']:>6}: joint={float(row['joint_acc'])*100:.1f}% R-marg={float(row['R_marginal'])*100:.1f}% C-marg={float(row['C_marginal'])*100:.1f}%")

   rel: joint=27.5% R-marg=45.0% C-marg=57.5%
   ctx: joint=40.8% R-marg=40.8% C-marg=100.0%
   R+C: joint=38.3% R-marg=41.7% C-marg=90.8%
 F+R+C: joint=35.8% R-marg=39.2% C-marg=91.7%


## 6. RC agreement/disagreement (18E, central: production logits/margins/norms)

In [6]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/agreement_disagreement.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"{row['split']:>9}: acc={float(row['acc'])*100:.1f}% margin={float(row['margin']):.3f} conf={float(row['confidence']):.3f} ent={float(row['entropy']):.3f}")
print('Q: does failure concentrate where heterogeneous evidence must be resolved?')

    agree: acc=100.0% margin=0.991 conf=0.995 ent=0.028
 disagree: acc=1.7% margin=0.985 conf=0.992 ent=0.035
Q: does failure concentrate where heterogeneous evidence must be resolved?


## 7. Logits and margins (18F: which component is favored; correlation only)

In [7]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/logit_margin.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"match C={float(row['matches_C_disagree'])*100:.1f}% match R={float(row['matches_R_disagree'])*100:.1f}% | "+
              f"rel/dis={float(row['rel_norm_disagree']):.3f} ctx/dis={float(row['ctx_norm_disagree']):.3f}")
print('Do NOT infer causality from correlation (§25).')

match C=98.3% match R=1.7% | rel/dis=7.609 ctx/dis=3.434
Do NOT infer causality from correlation (§25).


## 8. Counterfactual evidence (18G: change-rate is the valid responsiveness test)

In [8]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/counterfactuals.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"{row['swap']}: flip={float(row['flip_rate'])*100:.1f}%")
print('Implied-match cannot distinguish (a C-predictor matches both); change-rate can.')
cf = p18['aggregates']['counterfactual']
print(f"change-rates: R-swap {cf['R_swap_response']*100:.1f}% vs C-swap {cf['C_swap_response']*100:.1f}% (control {cf['control_same_response']*100:.1f}%)")

R_swap_flip: flip=98.3%
C_swap_flip: flip=96.6%
control_same_flip: flip=3.4%
R_swap_response: flip=0.0%
C_swap_response: flip=94.9%
control_same_response: flip=5.1%
Implied-match cannot distinguish (a C-predictor matches both); change-rate can.
change-rates: R-swap 7.3% vs C-swap 90.9% (control 7.1%)


## 9. Diagnostic heads (18H: linear vs nonlinear on frozen R+C)

In [9]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/diagnostic_heads.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print({k: v for k,v in row.items() if k != 'seed'})
print('Linear ok -> production limitation. Nonlinear-only -> nonlinear combination. Both fail -> deeper.')

{'head': 'linear_RC', 'RC': '0.5083333253860474', 'disagree_RC': '0.016949152573943138', 'params': '98', 'FRC': ''}
{'head': 'nonlinear_RC', 'RC': '0.4749999940395355', 'disagree_RC': '0.0', 'params': '1226', 'FRC': ''}
{'head': 'linear_FRC', 'RC': '', 'disagree_RC': '', 'params': '0', 'FRC': '0.800000011920929'}
{'head': 'nonlinear_FRC', 'RC': '', 'disagree_RC': '', 'params': '0', 'FRC': '0.800000011920929'}
Linear ok -> production limitation. Nonlinear-only -> nonlinear combination. Both fail -> deeper.


## 10. Objective diagnosis (18I: feasible rules, not label-oracles)

In [10]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/objective_diagnosis.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"model tr/te={float(row['model_train_RC'])*100:.1f}/{float(row['model_test_RC'])*100:.1f}% | "+
              f"R-rule(tr/te)={float(row['rule_R_train_RC'])*100:.1f}/{float(row['rule_R_test_RC'])*100:.1f}% | "+
              f"C-rule(tr/te)={float(row['rule_C_train_RC'])*100:.1f}/{float(row['rule_C_test_RC'])*100:.1f}%")
print('R-rule is label-oracle (sr erased); judge shortcut on the feasible C-rule.')

model tr/te=52.5/51.7% | R-rule(tr/te)=100.0/100.0% | C-rule(tr/te)=52.5/50.8%
R-rule is label-oracle (sr erased); judge shortcut on the feasible C-rule.


## 11. RC/FRC semantic contrast (18J: erasure proof + parity algebra)

In [11]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/rc_frc_semantics.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"parity={row['parity_fact']} C-match={float(row['production_matches_C_rule'])*100:.1f}% | "+
              f"align R/RC/FRC={float(row['align_R'])*100:.1f}/{float(row['align_RC'])*100:.1f}/{float(row['align_FRC'])*100:.1f}%")
print('align: autocorr-stat vs sr. High on R validates it; chance on RC/FRC proves erasure.')
print('Construction: cand added to ch0, then channels 0:3 overwritten (verified by reading).')

parity=True C-match=99.2% | align R/RC/FRC=98.3/40.8/51.7%
align: autocorr-stat vs sr. High on R validates it; chance on RC/FRC proves erasure.
Construction: cand added to ch0, then channels 0:3 overwritten (verified by reading).


## 12. Shuffled controls (18K: no accidental correlation)

In [12]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/shuffled_controls.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"label-shuffled={float(row['label_shuffled_RC'])*100:.1f}% R-permuted={float(row['R_permuted_RC'])*100:.1f}% (must be ~chance)")

label-shuffled=59.2% R-permuted=54.2% (must be ~chance)


## 13. Intervention gate (18L: two converging observations required)

In [13]:
print('gate:', p18['gate'])
print('Default is NO INTERVENTION.')

gate: {'passed': False, 'mechanism': 'none', 'candidate': 'none', 'evidence': ['18L requires two converging observations; not satisfied'], 'detail': 'H2=NOT SUPPORTED, H3=SUPPORTED, H5=NOT SUPPORTED, H6=SUPPORTED'}
Default is NO INTERVENTION.


## 14. Optional minimal intervention (18M: only if gate passes)

In [14]:
print('intervention record:', p18['aggregates']['intervention'])
print('minimal intervention:', p18['minimal_intervention'])

intervention record: {'tested': False, 'reason': '18L requires two converging observations; not satisfied'}
minimal intervention: {'intervention': 'none', 'outcome': 'NO INTERVENTION (default)', 'detail': '18L requires two converging observations; not satisfied'}


## 15. RC/FRC evaluation (18N primary criteria)

In [15]:
import csv
for row in csv.DictReader(open('../results/metrics/phase18_component_composition/rc_frc_results.csv')):
    if row['seed'] == str(p18['per_seed_results'][0]['seed']):
        print(f"{row['condition']:>9}: R={float(row['R'])*100:.1f}% RC={float(row['RC'])*100:.1f}% FRC={float(row['FRC'])*100:.1f}%")

    graph: R=90.8% RC=51.7% FRC=48.3%
 baseline: R=50.0% RC=50.8% FRC=80.0%
   depth3: R=60.0% RC=51.7% FRC=80.0%


## 16. Empirical ceiling (18O: never redefined)

In [16]:
c = p18['aggregates']['ceiling']
print(f"{c['old_ceiling_mixed_mean']*100:.1f}% -> {c['new_ceiling_mixed_mean']*100:.1f}% ({c['new_minus_old_mixed']*100:+.1f}pp)")

66.3% -> 67.3% (+1.0pp)


## 17. H1-H8 (re-derived programmatically)

In [17]:
from neuroforge.evaluation.phase18_component_composition import build_phase18_hypotheses
hyps = build_phase18_hypotheses(p18['aggregates'])
for h in sorted(hyps):
    print(f"{h}: {hyps[h]['status']}")
    print(f"    {hyps[h]['evidence']}")
assert all(hyps[h]['status'] == p18['hypotheses'][h]['status'] for h in hyps)
print('stored verdicts match fresh derivation: OK')

H1: PARTIALLY SUPPORTED
    Only one component available (R 48.1%, C 99.4%).
H2: NOT SUPPORTED
    Joint (R&C) accuracy only 43.1% on RC.
H3: SUPPORTED
    RC agree 99.5% vs disagree 1.2% (gap 98.3pp).
H4: SUPPORTED
    On RC-disagree, predictions match C 98.8% vs R 1.2%: systematic C-favoring.
H5: NOT SUPPORTED
    Both fail RC-disagree (linear 6.1%, nonlinear 4.5%): deeper than expressivity.
H6: SUPPORTED
    Model train RC 50.8% ≈ feasible C-rule 50.0%, behavior matches C-rule 99.2% (R-rule is label-oracle/infeasible).
H7: SUPPORTED
    Input-level erasure demonstrated (autocorr-stat recovers sr 97.8% on R vs 48.1% on RC; construction overwrites channels 0:3 after adding cand). RC-disagree labels equal R identically yet R is unobservable: composition untestable as constructed.
H8: NOT TESTED
    18L requires two converging observations; not satisfied
stored verdicts match fresh derivation: OK


## 18. Final CASE (programmatic)

In [18]:
from neuroforge.evaluation.phase18_component_composition import select_phase18_case
case, label = select_phase18_case(hyps, p18['aggregates'])
print(f'Programmatic verdict: {case} — {label}')
assert case == p18['verdict_case']
print(f"Minimal intervention: {p18['minimal_intervention']['intervention']} ({p18['minimal_intervention']['outcome']})")

Programmatic verdict: CASE D — RC/FRC behavior is primarily explained by task semantics (input-level erasure)
Minimal intervention: none (NO INTERVENTION (default))


## 19. Next-step recommendation (programmatic)

In [19]:
from neuroforge.evaluation.phase18_component_composition import recommendation_for_case
print('Recommendation:', recommendation_for_case(p18['verdict_case']))
print()
print('Loaded (not typed):')
s = p18['aggregates']['semantics']
print(f"  erasure={s['erasure_demonstrated']}, align R/RC={[round(s[k]*100,1) for k in ('align_R','align_RC')]}")
print(f"  disagree={p18['aggregates']['agreement']['disagree_acc']*100:.1f}%, C-favor={p18['aggregates']['favor']['disagree_pred_matches_C']*100:.1f}%")
print(f"  R-swap response={p18['aggregates']['counterfactual']['R_swap_response']*100:.1f}%, C-swap={p18['aggregates']['counterfactual']['C_swap_response']*100:.1f}%")

Recommendation: Accept the semantic boundary; do not relitigate parity-tie labels with architecture.

Loaded (not typed):
  erasure=True, align R/RC=[97.8, 48.1]
  disagree=1.2%, C-favor=98.8%
  R-swap response=7.3%, C-swap=90.9%
